# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedumer1941/Flyrank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring**



## 1. Two paper findings + my methodology questions

**Finding A: "Freshness multiplies existing quality"** — The paper reports that 365+ day content refreshed within 30 days shows a 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4,039).

**My question:** The label here is a **health-score change** (pre-refresh vs post-refresh), not the same `is_declining` label I use. The validation compares a before/after snapshot within the same content items, which is a reasonable paired comparison for a freshness study. However, the 57x impression jump comes from comparing mean impressions across two groups of pages (refreshed vs not refreshed), not from a controlled experiment. Pages that got refreshed may have been the more promising ones to begin with — selection bias is not addressed. A safer claim would say "refreshed 365+ day content observed 57x more impressions than similar non-refreshed content in this portfolio."

**Finding B: "Weighted portfolio CTR declines sharply as visibility moves away from the top of the results"** — CTR drops as position tiers go from top_3 to deep.

**My question:** The label is **CTR itself**, measured across position tiers. This is a direct aggregate comparison, so the validation is straightforward: split the portfolio by position tier and measure CTR. The finding is well-supported. However, the paper doesn't control for query intent — navigational queries at position 20 have essentially zero CTR, while informational queries at position 20 still get some clicks. A stratified analysis by intent would strengthen the claim. For my Lane 2 model, this finding supports using `avg_position` and `ctr` as predictive features (which I do), but also warns me that position-based features interact with intent in ways a simple model might not capture.

## 2. My model under an honest split (before/after)



My Week-5 Random Forest was already evaluated on a **client-holdout split** (20% of clients held out, no pages from test clients seen during training). This is already an honest split. Below I compare what happens if I use a **naive random split** (which leaks client information) vs the honest client-holdout split.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# --- Load and prepare data (same as ML-08) ---
URL = "https://raw.githubusercontent.com/ahmedumer1941/Flyrank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Build features
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
X_num = df[numeric_features].copy()
for col in ['search_volume', 'competition', 'cpc', 'word_count', 'char_count']:
    X_num[col] = X_num.groupby('content_type')[col].transform(lambda s: s.fillna(s.median()))
    X_num[col] = X_num[col].fillna(X_num[col].median())
X_num.loc[X_num['avg_position'] == 0, 'avg_position'] = X_num['avg_position'].median()

# Engineered
X_num['ctr_gap'] = (X_num['ctr'].max() - X_num['ctr']) / (X_num['ctr'].max() - X_num['ctr'].min() + 1e-6)
X_num['staleness_w'] = X_num['days_since_last_update'] / (X_num['content_age_days'] + 1e-6)
X_num['eng_per_session'] = X_num['engaged_sessions_90d'] / (X_num['sessions_90d'] + 1e-6)

cat_feats = ['content_type', 'main_intent', 'competition_level',
             'age_tier', 'freshness_tier', 'impression_tier',
             'word_count_tier', 'position_tier']
X_cat = pd.get_dummies(df[cat_feats], drop_first=True)
X = pd.concat([X_num, X_cat], axis=1).values
y = df['is_declining'].values
clients = df['client_id'].values

rf = RandomForestClassifier(n_estimators=100, max_depth=12,
                             random_state=42, class_weight='balanced', n_jobs=-1)

def precision_at_k(y_true, scores, k):
    top_k = np.argsort(scores)[-k:][::-1]
    return y_true[top_k].mean()

print("=== HONEST SPLIT COMPARISON ===\n")

# 1) Client-holdout split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=clients))
rf.fit(X[train_idx], y[train_idx])
scores_honest = rf.predict_proba(X[test_idx])[:, 1]
auc_honest = roc_auc_score(y[test_idx], scores_honest)
p50_honest = precision_at_k(y[test_idx], scores_honest, 50)
print(f"Client-Holdout (honest): {len(set(clients[train_idx]))} train clients, "
      f"{len(set(clients[test_idx]))} test clients ({len(test_idx)} rows)")
print(f"  Random Forest ROC-AUC: {auc_honest:.3f} | Precision@50: {p50_honest:.3f}\n")

# 2) Random split (naive — no group constraint)
train_idx2, test_idx2 = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
rf.fit(X[train_idx2], y[train_idx2])
scores_naive = rf.predict_proba(X[test_idx2])[:, 1]
auc_naive = roc_auc_score(y[test_idx2], scores_naive)
p50_naive = precision_at_k(y[test_idx2], scores_naive, 50)
print(f"Random-Split (naive): same data, same model, no group constraint")
print(f"  Random Forest ROC-AUC: {auc_naive:.3f} | Precision@50: {p50_naive:.3f}\n")

print("=== WHAT THIS MEANS ===")
print(f"The naive random split overestimates performance by +{auc_naive-auc_honest:.3f} ROC-AUC "
      f"and +{p50_naive-p50_honest:.3f} Precision@50.")
print("This is because the random split lets the model memorize client-specific patterns")
print("(same client's pages in both train and test), which inflates metrics.")
print(f"The client-holdout split is the honest number: {auc_honest:.3f} ROC-AUC / "
      f"{p50_honest:.3f} Precision@50.")

## 3. Leakage audit


The final feature set used in my model (ML-08) is the same as ML-05's: 22 numeric features
(+ 5 engineered) plus 8 one-hot encoded categorical features. Below I run the same 5 leakage
checks on the final training data.

In [1]:
# Build the FINAL feature set (same as ML-08)
final_numeric = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

final_cat = [
    'content_type', 'main_intent', 'competition_level',
    'age_tier', 'freshness_tier', 'impression_tier',
    'word_count_tier', 'position_tier'
]

print("=== LEAKAGE AUDIT (Final Feature Set) ===\n")

# Check 1: Label-derived
label_derived = ['trend_direction', 'trend_pct', 'is_declining']
leaked = [c for c in label_derived if c in final_numeric]
print(f"Check 1 \u2014 Label-derived columns in features: {leaked if leaked else 'NONE \u2713'}")
print(f"  Verified: trend_direction, trend_pct, is_declining NOT in feature matrix\n")

# Check 2: Future windows
future = ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
          'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
leaked2 = [c for c in future if c in final_numeric]
print(f"Check 2 \u2014 Future-window columns (last_30d, prev_30d): {leaked2 if leaked2 else 'NONE \u2713'}")
print(f"  Verified: these windowed columns were deliberately excluded from numeric_features\n")

# Check 3: IDs
print(f"Check 3 \u2014 content_id/client_id used as features: NO \u2713")
print(f"  Both identifiers used only for grouping/join, never in X matrix\n")

# Check 4: Missingness
print(f"Check 4 \u2014 Missingness encoding does not silently encode content_type \u2713")
print(f"  Using per-content-type median imputation instead of blind fillna(0)\n")

# Check 5: Provider/model
print(f"Check 5 \u2014 Provider/model columns excluded: YES \u2713")
print(f"  provider_used and model_used are NOT in the feature set\n")

print("=== LEAKAGE CHECK PASSED ===")
print("No label-derived, future-window, identifier, or product-flag columns in final features.")

=== LEAKAGE AUDIT (Final Feature Set) ===

Check 1 — Label-derived columns in features: NONE ✓
  Verified: trend_direction, trend_pct, is_declining NOT in feature matrix

Check 2 — Future-window columns (last_30d, prev_30d): NONE ✓
  Verified: these windowed columns were deliberately excluded from numeric_features

Check 3 — content_id/client_id used as features: NO ✓
  Both identifiers used only for grouping/join, never in X matrix

Check 4 — Missingness encoding does not silently encode content_type ✓
  Using per-content-type median imputation instead of blind fillna(0)

Check 5 — Provider/model columns excluded: YES ✓
  provider_used and model_used are NOT in the feature set

=== LEAKAGE CHECK PASSED ===
No label-derived, future-window, identifier, or product-flag columns in final features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [2]:
original = """
ORIGINAL (bold):
  "Our Random Forest model achieves 0.758 ROC-AUC and 0.800 Precision@50,
   significantly outperforming the baseline heuristic (0.641 ROC-AUC)."
"""

rewrite = """
REWRITE (safe, evidence-grounded):
  "On this dataset and client-holdout evaluation, the Random Forest model
   produced an observed ROC-AUC of 0.758 and Precision@50 of 0.800. Under
   the same split, the hand-crafted baseline score measured 0.641 ROC-AUC.
   This directional improvement suggests the learned model weights signals
   more effectively than the fixed heuristic for this particular slice.
   These are decision-support numbers, not guarantees of production performance."
"""

changes = """
Changes made:
  - "achieves" -> "produced an observed" (measured, not inherent)
  - "significantly outperforming" -> "directional improvement" (measured, not causal)
  - Added "on this dataset and client-holdout evaluation" (context boundary)
  - Added "for this particular slice" (generalization caveat)
  - Added decision-support disclaimer
"""

print("=== CLAIM REWRITE ===")
print(original)
print(rewrite)
print(changes)

=== CLAIM REWRITE ===

ORIGINAL (bold):
  "Our Random Forest model achieves 0.758 ROC-AUC and 0.800 Precision@50,
   significantly outperforming the baseline heuristic (0.641 ROC-AUC)."


REWRITE (safe, evidence-grounded):
  "On this dataset and client-holdout evaluation, the Random Forest model
   produced an observed ROC-AUC of 0.758 and Precision@50 of 0.800. Under
   the same split, the hand-crafted baseline score measured 0.641 ROC-AUC.
   This directional improvement suggests the learned model weights signals
   more effectively than the fixed heuristic for this particular slice.
   These are decision-support numbers, not guarantees of production performance."


Changes made:
  - "achieves" -> "produced an observed" (measured, not inherent)
  - "significantly outperforming" -> "directional improvement" (measured, not causal)
  - Added "on this dataset and client-holdout evaluation" (context boundary)
  - Added "for this particular slice" (generalization caveat)
  - Added decision

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.